## Analysis for all surface layer classes/groups

In [2]:
import os
import glob
import numpy as np
import pandas as pd

# -------------------------------------------------------------------
# 1. Paths
# -------------------------------------------------------------------
base_dir = "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/pixel_mean"
out_dir = "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/surf_layer_results"
os.makedirs(out_dir, exist_ok=True)

# -------------------------------------------------------------------
# 2. Utilities for incremental aggregation
# -------------------------------------------------------------------
def init_acc():
    # N, sum_x, sum_y, sum_diff, sum_diff2, sum_x2, sum_y2, sum_xy
    return dict(N=0, sum_x=0.0, sum_y=0.0,
                sum_diff=0.0, sum_diff2=0.0,
                sum_x2=0.0, sum_y2=0.0, sum_xy=0.0)

def update_acc(acc, x, y):
    """
    x, y are 1D numpy arrays (valid pairs only)
    Updates the accumulator in-place.
    """
    if len(x) == 0:
        return acc

    diff = x - y

    acc["N"] += len(x)
    acc["sum_x"] += x.sum()
    acc["sum_y"] += y.sum()
    acc["sum_diff"] += diff.sum()
    acc["sum_diff2"] += (diff ** 2).sum()
    acc["sum_x2"] += (x ** 2).sum()
    acc["sum_y2"] += (y ** 2).sum()
    acc["sum_xy"] += (x * y).sum()

    return acc

def finalize_metrics(acc):
    """
    From accumulator to bias, ubRMSD, R, N.
    """
    N = acc["N"]
    if N == 0:
        return np.nan, np.nan, np.nan, 0

    sum_x = acc["sum_x"]
    sum_y = acc["sum_y"]
    sum_diff = acc["sum_diff"]
    sum_diff2 = acc["sum_diff2"]
    sum_x2 = acc["sum_x2"]
    sum_y2 = acc["sum_y2"]
    sum_xy = acc["sum_xy"]

    # Mean diff
    mean_diff = sum_diff / N
    bias = mean_diff

    # E[(diff - mean_diff)^2] = E[diff^2] - mean_diff^2
    mean_diff2 = sum_diff2 / N
    var_ubrmsd = mean_diff2 - mean_diff ** 2
    if var_ubrmsd < 0:
        var_ubrmsd = 0.0
    ubrmsd = np.sqrt(var_ubrmsd)

    # Correlation R
    # cov_xy = E[xy] - E[x]E[y]
    Ex = sum_x / N
    Ey = sum_y / N
    Ex2 = sum_x2 / N
    Ey2 = sum_y2 / N
    Exy = sum_xy / N

    var_x = Ex2 - Ex ** 2
    var_y = Ey2 - Ey ** 2

    if var_x <= 0 or var_y <= 0:
        R = np.nan
    else:
        cov_xy = Exy - Ex * Ey
        R = cov_xy / np.sqrt(var_x * var_y)

    return bias, ubrmsd, R, N

# -------------------------------------------------------------------
# 3. Main accumulator dictionaries per grouping
# -------------------------------------------------------------------
acc_temp = {}   # key: temp_class
acc_lc = {}     # key: land_cover_group
acc_climate = {}# key: climate_group
acc_elev = {}   # key: elevation_class

# -------------------------------------------------------------------
# 4. Process files one by one
# -------------------------------------------------------------------
csv_paths = glob.glob(os.path.join(base_dir, "**", "*.csv"), recursive=True)
if not csv_paths:
    raise RuntimeError(f"No CSV files found under {base_dir}")

for i, csv_path in enumerate(csv_paths, start=1):
    try:
        df = pd.read_csv(csv_path, low_memory=False)
    except Exception as e:
        print(f"Failed to read {csv_path}: {e}")
        continue

    # Only proceed if the needed columns exist
    if "ts_era5" not in df.columns or "ts_reference" not in df.columns:
        continue

    # Ensure numeric for x, y
    df["ts_era5"] = pd.to_numeric(df["ts_era5"], errors="coerce")
    df["ts_reference"] = pd.to_numeric(df["ts_reference"], errors="coerce")

    # Valid pairs
    mask_valid = np.isfinite(df["ts_era5"].values) & np.isfinite(df["ts_reference"].values)
    if not mask_valid.any():
        continue

    # For grouping columns, work on the valid subset only
    df_valid = df.loc[mask_valid].copy()

    # ----------------- temperature class -----------------
    if "temp_class" in df_valid.columns:
        for cls, sub in df_valid.groupby("temp_class"):
            if cls == "ST-Unknown":
                continue
            x = sub["ts_era5"].values
            y = sub["ts_reference"].values
            if cls not in acc_temp:
                acc_temp[cls] = init_acc()
            acc_temp[cls] = update_acc(acc_temp[cls], x, y)

    # ----------------- land cover group ------------------
    if "land_cover_group" in df_valid.columns:
        for lc, sub in df_valid.groupby("land_cover_group"):
            x = sub["ts_era5"].values
            y = sub["ts_reference"].values
            if lc not in acc_lc:
                acc_lc[lc] = init_acc()
            acc_lc[lc] = update_acc(acc_lc[lc], x, y)

    # ----------------- climate group ---------------------
    if "climate_group" in df_valid.columns:
        for cg, sub in df_valid.groupby("climate_group"):
            x = sub["ts_era5"].values
            y = sub["ts_reference"].values
            if cg not in acc_climate:
                acc_climate[cg] = init_acc()
            acc_climate[cg] = update_acc(acc_climate[cg], x, y)

    # ----------------- elevation class -------------------
    if "elevation_class" in df_valid.columns:
        for ec, sub in df_valid.groupby("elevation_class"):
            x = sub["ts_era5"].values
            y = sub["ts_reference"].values
            if ec not in acc_elev:
                acc_elev[ec] = init_acc()
            acc_elev[ec] = update_acc(acc_elev[ec], x, y)

    if i % 50 == 0:
        print(f"Processed {i} files...")

print("Finished processing all files.")

# -------------------------------------------------------------------
# 5. Convert accumulators to DataFrames and save
# -------------------------------------------------------------------
def acc_to_df(acc_dict, group_name):
    rows = []
    for g, acc in sorted(acc_dict.items(), key=lambda kv: str(kv[0])):
        bias, ubrmsd, R, N = finalize_metrics(acc)
        rows.append(dict(
            group=g,
            bias=bias,
            ubrmsd=ubrmsd,
            R=R,
            N=N
        ))
    df_out = pd.DataFrame(rows)
    if not df_out.empty:
        df_out["bias"] = df_out["bias"].round(2)
        df_out["ubrmsd"] = df_out["ubrmsd"].round(2)
        df_out["R"] = df_out["R"].round(3)
    df_out.rename(columns={"group": group_name}, inplace=True)
    return df_out

# 5.1 Temperature classes
temp_stats = acc_to_df(acc_temp, "temp_class")
temp_path = os.path.join(out_dir, "era5_L1_metrics_by_temp_class.csv")
temp_stats.to_csv(temp_path, index=False)
print(f"Saved temperature-class metrics to: {temp_path}")

# 5.2 Land-cover groups
lc_stats = acc_to_df(acc_lc, "land_cover_group")
lc_path = os.path.join(out_dir, "era5_L1_metrics_by_land_cover_group.csv")
lc_stats.to_csv(lc_path, index=False)
print(f"Saved land-cover metrics to: {lc_path}")

# 5.3 Climate groups
climate_stats = acc_to_df(acc_climate, "climate_group")
climate_path = os.path.join(out_dir, "era5_L1_metrics_by_climate_group.csv")
climate_stats.to_csv(climate_path, index=False)
print(f"Saved climate metrics to: {climate_path}")

# 5.4 Elevation classes
elev_stats = acc_to_df(acc_elev, "elevation_class")
elev_path = os.path.join(out_dir, "era5_L1_metrics_by_elevation_class.csv")
elev_stats.to_csv(elev_path, index=False)
print(f"Saved elevation metrics to: {elev_path}")

# Optional quick preview
if not temp_stats.empty:
    print("\nTemperature-class metrics (preview):")
    print(temp_stats.to_markdown(index=False))


Processed 50 files...
Processed 100 files...
Processed 150 files...
Processed 200 files...
Processed 250 files...
Processed 300 files...
Processed 350 files...
Processed 400 files...
Processed 450 files...
Processed 500 files...
Processed 550 files...
Processed 600 files...
Processed 650 files...
Processed 700 files...
Processed 750 files...
Processed 800 files...
Processed 850 files...
Finished processing all files.
Saved temperature-class metrics to: /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/surf_layer_results/era5_L1_metrics_by_temp_class.csv
Saved land-cover metrics to: /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/surf_layer_results/era5_L1_metrics_by_land_cover_group.csv
Saved climate metrics to: /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/surf_layer_results/era5_L1_metrics_by_climate_group.csv
Saved elevation metrics to: /home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/surf_layer_results/era5_L1_m

## Checking the correlation and ubRMSD between the in_situ and products during day and night

In [2]:
import numpy as np
import pandas as pd

def compute_phase_lag(ts1, ts2):
    """
    Returns the lag (in time steps) where ts2 best matches ts1.
    Positive lag: ts2 lags ts1 (ts1 leads).
    Negative lag: ts2 leads ts1.
    """
    ts1 = np.asarray(ts1)
    ts2 = np.asarray(ts2)
    # Remove NaNs
    mask = np.isfinite(ts1) & np.isfinite(ts2)
    ts1 = ts1[mask]
    ts2 = ts2[mask]
    if len(ts1) == 0 or len(ts2) == 0:
        return np.nan
    # Subtract mean for unbiased cross-correlation
    ts1 = ts1 - np.mean(ts1)
    ts2 = ts2 - np.mean(ts2)
    corr = np.correlate(ts1, ts2, mode='full')
    lag = np.argmax(corr) - (len(ts2) - 1)
    return lag, corr

# Example usage for a single file:
csv_path = "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/pixel_mean/ARM/sparse/Anthony_insitu_era5_soil_temperature_level1.csv"
df = pd.read_csv(csv_path)
lag, corr = compute_phase_lag(df["ts_reference"], df["ts_era5"])
print(f"Phase lag (in time steps): {lag}")

Phase lag (in time steps): 2


In [16]:
import pandas as pd
import numpy as np

# Load the CSV
csv_path = "/home/aminr/Desktop/Master_Thesis/Global_scale/surface_layer/era5/pixel_mean/TERENO/sparse/Schoeneseiffen_insitu_era5_soil_temperature_level1.csv"
df = pd.read_csv(csv_path)

# Combine date and time into a datetime column
df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'])
df['day'] = df['datetime'].dt.date
df['hour'] = df['datetime'].dt.hour

# Select one month (e.g., January 2019)
month = 1
year = 2019
df_month = df[(df['datetime'].dt.month == month) & (df['datetime'].dt.year == year)]

In [17]:
def compute_phase_lag(ts1, ts2):
    ts1 = np.asarray(ts1)
    ts2 = np.asarray(ts2)
    mask = np.isfinite(ts1) & np.isfinite(ts2)
    ts1 = ts1[mask]
    ts2 = ts2[mask]
    if len(ts1) == 0 or len(ts2) == 0:
        return np.nan
    ts1 = ts1 - np.mean(ts1)
    ts2 = ts2 - np.mean(ts2)
    corr = np.correlate(ts1, ts2, mode='full')
    lag = np.argmax(corr) - (len(ts2) - 1)
    return lag

results = []

for day, group in df_month.groupby('day'):
    lag = compute_phase_lag(group['ts_reference'], group['ts_era5'])
    # Find the hour of max and min for both series
    idx_max_ref = group['ts_reference'].idxmax()
    idx_min_ref = group['ts_reference'].idxmin()
    idx_max_era5 = group['ts_era5'].idxmax()
    idx_min_era5 = group['ts_era5'].idxmin()
    results.append({
        'day': day,
        'phase_lag_hours': lag,
        'ts_reference_max': group.loc[idx_max_ref, 'ts_reference'],
        'ts_reference_max_time': group.loc[idx_max_ref, 'time'],
        'ts_reference_min': group.loc[idx_min_ref, 'ts_reference'],
        'ts_reference_min_time': group.loc[idx_min_ref, 'time'],
        'ts_era5_max': group.loc[idx_max_era5, 'ts_era5'],
        'ts_era5_max_time': group.loc[idx_max_era5, 'time'],
        'ts_era5_min': group.loc[idx_min_era5, 'ts_era5'],
        'ts_era5_min_time': group.loc[idx_min_era5, 'time'],
    })

# Convert to DataFrame for pretty printing
results_df = pd.DataFrame(results)
print(results_df)

           day  phase_lag_hours  ts_reference_max ts_reference_max_time  \
0   2019-01-01                0           277.750                 16:00   
1   2019-01-02                0           277.283                 00:00   
2   2019-01-03              -16           276.317                 00:00   
3   2019-01-04                0           276.217                 13:00   
4   2019-01-05                0           276.350                 19:00   
5   2019-01-06               -2           276.583                 14:00   
6   2019-01-07                0           276.717                 23:00   
7   2019-01-08                0           276.950                 04:00   
8   2019-01-09               14           276.283                 00:00   
9   2019-01-10               10           275.717                 00:00   
10  2019-01-11                0           275.583                 23:00   
11  2019-01-12                0           276.183                 21:00   
12  2019-01-13           

In [18]:
import numpy as np

def match_stats(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() == 0:
        return np.nan, np.nan
    x = x[mask]
    y = y[mask]
    # Correlation
    corr = np.corrcoef(x, y)[0, 1]
    # ubRMSD
    diff = x - y
    ubrmsd = np.sqrt(np.mean((diff - np.mean(diff))**2))
    return corr, ubrmsd

results = []

for day, group in df_month.groupby('day'):
    # Night: 0-5 and 20-23, Day: 6-19
    night_mask = group['hour'].isin([0,1,2,3,4,5,20,21,22,23])
    day_mask = group['hour'].isin(range(6,20))
    corr_night, ubrmsd_night = match_stats(group.loc[night_mask, 'ts_reference'], group.loc[night_mask, 'ts_era5'])
    corr_day, ubrmsd_day = match_stats(group.loc[day_mask, 'ts_reference'], group.loc[day_mask, 'ts_era5'])
    results.append({
        'day': day,
        'corr_night': corr_night,
        'ubrmsd_night': ubrmsd_night,
        'corr_day': corr_day,
        'ubrmsd_day': ubrmsd_day
    })

results_df = pd.DataFrame(results)
print(results_df)

           day  corr_night  ubrmsd_night  corr_day  ubrmsd_day
0   2019-01-01    0.887781      0.105075  0.711212    0.118793
1   2019-01-02    0.733383      0.355908 -0.066322    0.510938
2   2019-01-03   -0.333068      0.174979  0.166853    0.448177
3   2019-01-04    0.654478      0.319660  0.443537    0.238804
4   2019-01-05    0.976010      0.742576  0.820742    0.444115
5   2019-01-06    0.671862      0.228858  0.896280    0.229699
6   2019-01-07    0.944589      0.237978  0.927829    0.231587
7   2019-01-08    0.923847      0.314246  0.599964    0.153455
8   2019-01-09    0.908655      0.286837 -0.426353    0.682317
9   2019-01-10    0.812102      0.116703 -0.504796    0.405790
10  2019-01-11    0.940515      0.299522  0.827913    0.288334
11  2019-01-12    0.983587      0.824561  0.911289    0.337957
12  2019-01-13    0.865258      0.057725  0.974966    0.302714
13  2019-01-14    0.992224      0.573002  0.695092    0.377820
14  2019-01-15    0.622847      0.510167  0.587675    0

In [19]:
print("Mean night ubRMSD:", results_df['ubrmsd_night'].mean())
print("Mean day ubRMSD:", results_df['ubrmsd_day'].mean())

Mean night ubRMSD: 0.40515960231748305
Mean day ubRMSD: 0.455927326572336


In [20]:
print("Mean night correlation:", results_df['corr_night'].mean())
print("Mean day correlation:", results_df['corr_day'].mean())

Mean night correlation: 0.4273051758617485
Mean day correlation: 0.2009233657536508
